# Bot de monitoreo geriátrico remoto



Descripción

El usuario es un cuidador (familiar o técnico de enfermería) encargado de un adulto mayor con hipertensión y diabetes.
Necesita llevar un diario clínico riguroso para enviárselo al médico a fin de mes.

El bot permite:
- Registrar presión arterial y glucosa en sangre múltiples veces al día.
- Confirmar la toma de medicamentos.
- Calcular promedios del período automáticamente.
- Activar una evaluación de riesgo que detecta valores peligrosos reiterados.
- Generar un gráfico temporal de glucosa con zona de peligro resaltada.
- Exportar un informe clínico listo para teleconsulta.

# Librerías

In [ ]:
# Instalación de dependencias
!pip install python-telegram-bot nest_asyncio matplotlib numpy -q

# Librerías estándar de Python
import os
import logging
import pickle
import re as _re
from io import BytesIO
from datetime import datetime

# Librerías de terceros (datos, gráficos y utilidades)
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pytz

# Librerías específicas para el Bot de Telegram
from telegram import (
    Update, 
    InputFile, 
    ReplyKeyboardMarkup, 
    ReplyKeyboardRemove
)
from telegram.ext import (
    ContextTypes, 
    ApplicationBuilder, 
    CommandHandler, 
    MessageHandler, 
    filters, 
    PicklePersistence
)

# Entorno de Google Colab y manejo de asincronía
from google.colab import userdata, drive
import nest_asyncio

# Configuración inicial e inicialización
nest_asyncio.apply()

# Fase 1 - Credenciales

In [ ]:
try:
    from google.colab import userdata
    TOKEN = userdata.get('TELEGRAM_TOKEN')
except ImportError:
    TOKEN = os.environ.get('TELEGRAM_TOKEN')

if TOKEN:
    logger.info("Token cargado correctamente.")
else:
    logger.error("Error: Token de Telegram no encontrado.")

# Fase 2 - Constantes, modelo de datos y helpers

## Constantes

In [ ]:
# Umbrales clínicos (segun OMS / ADA)
# Definen el rango normal de valores
UMBRAL_SISTOLICA_MAX  = 140
UMBRAL_SISTOLICA_MIN  = 110
UMBRAL_DIASTOLICA_MAX = 90
UMBRAL_DIASTOLICA_MIN = 60
UMBRAL_GLUCOSA_MAX    = 180
UMBRAL_GLUCOSA_MIN    = 70
ALARMAS_NECESARIAS    = 3

# Síntomas comunes
SINTOMAS_COMUNES = [
    "Mareos", "Fatiga", "Visión borrosa",
    "Dolor de cabeza", "Náuseas", "Palpitaciones",
]

# Medicamentos comunes
MEDICAMENTOS_DICT = {
    'losartan'     : {'categoria': 'hipertension'},
    'enalapril'    : {'categoria': 'hipertension'},
    'amlodipino'   : {'categoria': 'hipertension'},
    'metformina'   : {'categoria': 'diabetes'},
    'insulina'     : {'categoria': 'diabetes'},
    'glibenclamida': {'categoria': 'diabetes'},
}

print("Constantes listas")

## Modelo de datos

In [ ]:
# Estado inicial del sistema
def inicializar_estado(context: ContextTypes.DEFAULT_TYPE):
    if 'pacientes' not in context.user_data:
        context.user_data['pacientes']       = {}
        context.user_data['paciente_activo'] = None
        context.user_data['estado']          = None
        context.user_data['temp']            = {}   # Almacén temporal multi-campo

def datos_paciente(context):
    return context.user_data['pacientes'][context.user_data['paciente_activo']]

def hay_paciente_activo(context):
    n = context.user_data.get('paciente_activo')
    return n is not None and n in context.user_data.get('pacientes', {})

def nombre_pac(context):
    return context.user_data.get('paciente_activo', '-')

def crear_paciente(context, nombre):
    nombre = nombre.strip().title()
    existe = nombre in context.user_data['pacientes']
    if not existe:
        context.user_data['pacientes'][nombre] = {
            'perfil'       : {},      # nombre, edad, diagnosticos, medicacion_perm, contacto_emergencia
            'presion'      : [],
            'glucosa'      : [],
            'medicamentos' : [],      # registros de toma {fecha,nombre,dosis,frecuencia}
            'sintomas'     : [],      # {fecha,sintoma,intensidad}
            'alertas'      : [],      # {fecha,tipo,mensaje,critica}
            'cuidador'     : None,    # chat_id del cuidador/familiar
        }
    return not existe, nombre

print("Modelo de datos listo")

## Helpers

In [ ]:
# Cache de zona horaria por sesión
def ahora():
    return datetime.now(pytz.timezone("America/Lima")).strftime('%d/%m %H:%M')

# Registro de eventos
def registrar_alerta(context, tipo, mensaje, critica=False):
    datos = datos_paciente(context)
    datos['alertas'].append({
        'fecha'  : ahora(),
        'tipo'   : tipo,
        'mensaje': mensaje,
        'critica': critica,
    })
    logger.warning(f"[ALERTA] {tipo} - {mensaje}")

print("Helpers listos")

# Fase 3 - Teclados y display

## Teclado (menú)

In [ ]:
# Teclados estáticos
def teclado_menu():
    return ReplyKeyboardMarkup([
        ["👤 Pacientes"],
        ["📋 Registrar datos"],
        ["📊 Informes"],
        ["🗑️ Limpiar registros"],
        ["🚪 Salir por ahora"],
    ], resize_keyboard=True)

def teclado_registrar():
    return ReplyKeyboardMarkup([
        ["🩺 Presión arterial", "🩸 Glucosa"],
        ["💊 Medicamento", "🤒 Síntomas"],
        ["⬅️ Volver al menú"],
    ], resize_keyboard=True)

def teclado_informes():
    return ReplyKeyboardMarkup([
        ["⚠️ Evaluación de riesgo"],
        ["📄 Generar informe completo"],
        ["👤 Ver perfil del paciente"],
        ["⬅️ Volver al menú"],
    ], resize_keyboard=True)

def teclado_pacientes(context):
    nombres = list(context.user_data.get('pacientes', {}).keys())
    botones = [[n] for n in nombres]
    botones += [["➕ Nuevo paciente"], ["⬅️ Volver al menú"]]
    return ReplyKeyboardMarkup(botones, resize_keyboard=True)

def teclado_medicamentos_dict():
    nombres = [m.capitalize() for m in MEDICAMENTOS_DICT]
    filas   = [nombres[i:i+2] for i in range(0, len(nombres), 2)]
    filas  += [["➕ Otro medicamento"], ["⬅️ Volver al menú"]]
    return ReplyKeyboardMarkup(filas, resize_keyboard=True, one_time_keyboard=True)

def teclado_intensidad():
    return ReplyKeyboardMarkup(
        [["🟡 Leve", "🟠 Moderado", "🔴 Severo"], ["⬅️ Cancelar"]],
        resize_keyboard=True, one_time_keyboard = True
    )

def teclado_sintomas():
    filas = [SINTOMAS_COMUNES[i:i+2] for i in range(0, len(SINTOMAS_COMUNES), 2)]
    filas += [["➕ Otro síntoma"], ["⬅️ Volver al menú"]]
    return ReplyKeyboardMarkup(filas, resize_keyboard= True, one_time_keyboard= True)

def teclado_confirmar_med():
    return ReplyKeyboardMarkup(
        [["✅ Registrar otro", "🏠 Volver al menú"]],
        resize_keyboard=True, one_time_keyboard = True
    )

def teclado_reposo():
    return ReplyKeyboardMarkup([
        ["📋 Seguir registrando"],
        ["👤 Cambiar paciente"],
        ["🏠 Volver al menú"],
        ["🚪 Salir por ahora"],
    ], resize_keyboard=True)

def teclado_despertar():
    return ReplyKeyboardMarkup([["🏠 Abrir menú"]], resize_keyboard=True)

print("Teclado listo")

## Display

In [ ]:
async def mostrar_menu(update, texto_extra="", context=None):
    pac = f"👤 Paciente activo: *{nombre_pac(context)}*\n" if context and hay_paciente_activo(context) else ""
    pre = (texto_extra + "\n\n") if texto_extra else ""
    await update.message.reply_text(
        pre + pac + "*¿Qué deseas hacer?*",
        parse_mode='Markdown', reply_markup=teclado_menu()
    )

async def mostrar_submenu_registrar(update, texto_extra="", context=None):
    pac = f"👤 *{nombre_pac(context)}*\n" if context and hay_paciente_activo(context) else ""
    pre = (texto_extra + "\n\n") if texto_extra else ""
    await update.message.reply_text(
        pre + pac + "¿Qué registras?",
        parse_mode='Markdown', reply_markup=teclado_registrar()
    )

async def mostrar_submenu_informes(update, texto_extra="", context=None):
    pac = f"👤 *{nombre_pac(context)}*\n" if context and hay_paciente_activo(context) else ""
    pre = (texto_extra + "\n\n") if texto_extra else ""
    await update.message.reply_text(
        pre + pac + "¿Qué consultas?",
        parse_mode='Markdown', reply_markup=teclado_informes()
    )

async def mostrar_submenu_pacientes(update, context, texto_extra=""):
    nombres = list(context.user_data.get('pacientes', {}).keys())
    lista   = "\n".join(f"  • {n}" for n in nombres) if nombres else "  (ninguno)"
    pre     = (texto_extra + "\n\n") if texto_extra else ""
    await update.message.reply_text(
        pre + f"*Gestión de pacientes*\nRegistrados:\n{lista}\n\nSelecciona o crea uno:",
        parse_mode='Markdown', reply_markup=teclado_pacientes(context)
    )

async def mostrar_reposo(update, confirmacion=""):
    msg = (confirmacion + "\n\n") if confirmacion else ""
    await update.message.reply_text(
        msg + "¿Qué deseas hacer ahora?",
        parse_mode='Markdown', reply_markup=teclado_reposo()
    )

print("Display listo")

## Guardo de datos en google drive

In [ ]:
# Usamos una carpeta local llamada 'datos_bot'
CARPETA_USUARIOS = './datos_bot/usuarios/'

def guardar_datos_en_drive(context, user_id):
    ruta_especifica = f'{CARPETA_USUARIOS}usuario_{user_id}.pkl'
    try:
        if not os.path.exists(CARPETA_USUARIOS):
            os.makedirs(CARPETA_USUARIOS)
            print(f"Estructura de carpetas creada: {CARPETA_USUARIOS}")

        with open(ruta_especifica, 'wb') as f:
            pickle.dump(context.user_data, f)
        print(f"Backup guardado: usuario_{user_id}.pkl")
    except Exception as e:
        print(f"Error al guardar datos: {e}")

def guardar_datos_en_local(context, user_id):
    ruta_usuario = f'{CARPETA_USUARIOS}usuario_{user_id}.pkl'
    try:
        if os.path.exists(ruta_usuario):
            with open(ruta_usuario, 'rb') as f:
                datos_recuperados = pickle.load(f)
                context.user_data.update(datos_recuperados)
            print(f"✅ Datos recuperados para el usuario {user_id}")
        else:
            print(f"ℹ️ No hay datos previos para {user_id}. Iniciando nuevo.")
    except Exception as e:
        print(f"❌ Error al cargar datos: {e}")

# Fase 4 - Lógica clínica

## Presión arterial

In [ ]:
# Registro de presion arterial (sistolica y diastolica) en mmHg
def guardar_presion(context, sistolica, diastolica, user_id):
    datos    = datos_paciente(context)
    fecha_registro    = ahora()
    registro = {'fecha': fecha_registro, 'sistolica': sistolica, 'diastolica': diastolica}
    datos['presion'].append(registro)
    total = len(datos['presion'])

    alerta_txt = ""
    # Motor de reglas: alerta inmediata
    if sistolica >= UMBRAL_SISTOLICA_MAX or diastolica >= UMBRAL_DIASTOLICA_MAX:
        alerta_txt = "\n⚠️ *ATENCIÓN:* Valor elevado - posible hipertensión."
        critica = sum(
            1 for r in datos['presion']
            if r['sistolica'] >= UMBRAL_SISTOLICA_MAX or r['diastolica'] >= UMBRAL_DIASTOLICA_MAX
        ) >= ALARMAS_NECESARIAS
        registrar_alerta(context, 'presion_alta',
            f"Presión {sistolica}/{diastolica} mmHg en {fecha_registro}", critica)
    elif sistolica < UMBRAL_SISTOLICA_MIN or diastolica < UMBRAL_DIASTOLICA_MIN:
        alerta_txt = "\n⚠️ *ATENCIÓN:* Valor bajo - posible hipotensión."
        registrar_alerta(context, 'presion_baja',
            f"Presión {sistolica}/{diastolica} mmHg en {fecha_registro}", False)

    logger.info(f"Presión {sistolica}/{diastolica} mmHg - {nombre_pac(context)}")
    guardar_datos_en_drive(context, user_id)
    return (
        f"*Presión registrada* [{fecha_registro}]\n"
        f"{sistolica}/{diastolica} mmHg\n"
        f"Total registros: {total}{alerta_txt}"
    )

print("Lógica de presión lista ")

## Glucosa

In [ ]:
# Registro de glucosa en mg/dL
def guardar_glucosa(context, valor, user_id):
    datos    = datos_paciente(context)
    fecha_registro    = ahora()
    registro = {'fecha': fecha_registro, 'valor': valor}
    datos['glucosa'].append(registro)
    total = len(datos['glucosa'])

    if valor < UMBRAL_GLUCOSA_MIN:
        estado_cl = "⚠️ Hipoglucemia"
        critica = sum(1 for r in datos['glucosa'] if r['valor'] < UMBRAL_GLUCOSA_MIN) >= 2
        registrar_alerta(context, 'glucosa_baja', f"Glucosa {valor} mg/dL en {fecha_registro}", critica)
    elif valor < 100:
        estado_cl = "✅ Normal en ayunas"
    elif valor < 140:
        estado_cl = "✅ Normal postprandial"
    elif valor < UMBRAL_GLUCOSA_MAX:
        estado_cl = "🟡 Elevada - Monitorear"
    else:
        estado_cl = "🔴 Hiperglucemia"
        critica = sum(1 for r in datos['glucosa'] if r['valor'] >= UMBRAL_GLUCOSA_MAX) >= ALARMAS_NECESARIAS
        registrar_alerta(context, 'glucosa_alta', f"Glucosa {valor} mg/dL en {fecha_registro}", critica)

    logger.info(f"Glucosa {valor} mg/dL - {nombre_pac(context)}")
    guardar_datos_en_drive(context, user_id)
    return (
        f"*Glucosa registrada* [{fecha_registro}]\n"
        f"{valor} mg/dL\n"
        f"Estado: {estado_cl}\n"
        f"Total registros: {total}"
    )

print("Lógica de glucosa lista")

## Medicamentos

In [ ]:
# Medicamentos (diccionario + libre)
def guardar_medicamento_libre(context, nombre, dosis, frecuencia, user_id):
    """
    Guarda cualquier medicamento con nombre, dosis y frecuencia libres.
    Retorna (total, fecha_registro)
    """
    datos = datos_paciente(context)
    fecha_registro = ahora()
    registro = {
        'fecha'      : fecha_registro,
        'nombre'     : nombre.strip().lower(),
        'dosis'      : dosis.strip(),
        'frecuencia' : frecuencia.strip(),
    }
    datos['medicamentos'].append(registro)
    total = len(datos['medicamentos'])
    logger.info(f"Medicamento '{nombre}' {dosis} - {frecuencia} - {nombre_pac(context)}")
    guardar_datos_en_drive(context, user_id)
    return total, fecha_registro

def guardar_medicamento_dict(context, nombre_canonico, user_id):
    """
    Guarda medicamento del diccionario controlado con dosis/frecuencia vacías.
    Retorna (total, fecha_registro).
    """
    if nombre_canonico not in MEDICAMENTOS_DICT:
        raise ValueError(f"Medicamento no reconocido: {nombre_canonico}")
    return guardar_medicamento_libre(context, nombre_canonico, "-", "-", user_id)

print("Lógica de medicamentos lista")

## Síntomas

In [ ]:
# Guardar síntomas con grado de dolor
def guardar_sintoma(context, sintoma, intensidad, user_id):
    datos = datos_paciente(context)
    fecha_registro = ahora()
    datos['sintomas'].append({'fecha': fecha_registro, 'sintoma': sintoma, 'intensidad': intensidad})
    if intensidad == "🔴 Severo":
        registrar_alerta(context, 'sintoma_severo',
            f"Síntoma severo: {sintoma} en {fecha_registro}", True)
    logger.info(f"Síntoma '{sintoma}' ({intensidad}) - {nombre_pac(context)}")
    guardar_datos_en_drive(context, user_id)
    return fecha_registro

print("Lógica de síntomas lista")

## Perfil del paciente

In [ ]:
# Mostrar el perfil del paciente
PERFIL_CAMPOS = [
    ('edad',               '¿Cuántos años tiene el paciente?'),
    ('diagnosticos',       '¿Cuáles son sus diagnósticos crónicos? (ej: HTA, Diabetes)'),
    ('medicacion_perm',    '¿Cuál es su medicación permanente?'),
    ('contacto_emergencia','¿Nombre y teléfono del contacto de emergencia?'),
]

def perfil_completo(context):
    """Retorna True si el perfil tiene todos los campos obligatorios."""
    perfil = datos_paciente(context).get('perfil', {})
    return all(k in perfil for k, _ in PERFIL_CAMPOS)

def texto_perfil(context):
    perfil = datos_paciente(context).get('perfil', {})
    nombre = nombre_pac(context)
    lineas = [f"*Perfil de {nombre}*", "━━━━━━━━━━━━━━━━━━━"]
    etiquetas = {
        'edad': '🎂 Edad',
        'diagnosticos': '🏥 Diagnósticos',
        'medicacion_perm': '💊 Medicación permanente',
        'contacto_emergencia': '📞 Contacto de emergencia',
    }
    for k, etiq in etiquetas.items():
        lineas.append(f"{etiq}: {perfil.get(k, '-')}")
    return "\n".join(lineas)

print("Lógica de perfil lista ")

## Evaluación de riesgo

In [ ]:
# Evaluación de riesgo
def evaluar_riesgo(context):
    datos  = datos_paciente(context)
    nombre = nombre_pac(context)
    rp     = datos['presion']
    rg     = datos['glucosa']

    if not rp and not rg:
        return "No hay datos suficientes. Registra mediciones primero."

    alertas      = []
    nivel_riesgo = "Bajo"

    if rp:
        ep = [r for r in rp if r['sistolica'] >= UMBRAL_SISTOLICA_MAX or r['diastolica'] >= UMBRAL_DIASTOLICA_MAX]
        pct = (len(ep) / len(rp)) * 100
        if len(ep) >= ALARMAS_NECESARIAS:
            nivel_riesgo = "Alto"
            fechas = ", ".join(r['fecha'] for r in ep[-3:])
            alertas.append(
                f"🔴 *HIPERTENSIÓN REITERADA*\n"
                f"  {len(ep)}/{len(rp)} mediciones superaron {UMBRAL_SISTOLICA_MAX}/{UMBRAL_DIASTOLICA_MAX} mmHg ({pct:.0f}%)\n"
                f"  Últimos episodios: {fechas}\n  Consulta médica urgente."
            )
        elif ep:
            nivel_riesgo = "Moderado"
            alertas.append(f"🟠 *Presión elevada ocasional* - {len(ep)} episodio(s). Vigilar.")

    if rg:
        hiper = [r for r in rg if r['valor'] >= UMBRAL_GLUCOSA_MAX]
        hipo  = [r for r in rg if r['valor'] < UMBRAL_GLUCOSA_MIN]
        if len(hiper) >= ALARMAS_NECESARIAS:
            nivel_riesgo = "Alto"
            pct = (len(hiper) / len(rg)) * 100
            fechas = ", ".join(r['fecha'] for r in hiper[-3:])
            alertas.append(
                f"🔴 *HIPERGLUCEMIA REITERADA*\n"
                f"  {len(hiper)}/{len(rg)} mediciones ≥ {UMBRAL_GLUCOSA_MAX} mg/dL ({pct:.0f}%)\n"
                f"  Últimos: {fechas}\n  Revisar dieta y dosis."
            )
        elif hiper:
            if nivel_riesgo != "Alto": nivel_riesgo = "Moderado"
            alertas.append(f"🟠 *Glucosa elevada ocasional* - {len(hiper)} episodio(s).")
        if len(hipo) >= 2:
            nivel_riesgo = "Alto"
            alertas.append(
                f"🔵 *HIPOGLUCEMIA REITERADA*\n"
                f"  {len(hipo)} episodio(s) < 70 mg/dL. Riesgo de pérdida de consciencia."
            )

    # Alerta de síntomas severos
    sint_sev = [s for s in datos.get('sintomas', []) if s['intensidad'] == "🔴 Severo"]
    if sint_sev:
        if nivel_riesgo != "Alto": nivel_riesgo = "Moderado"
        alertas.append(f"🔴 *Síntomas severos registrados* - {len(sint_sev)} episodio(s).")

    iconos = {"Bajo": "🟢", "Moderado": "🟠", "Alto": "🔴"}
    msg = (
        f"*Evaluación de riesgo - {nombre}*\n"
        f"━━━━━━━━━━━━━━━━━━━\n"
        f"Nivel global: {iconos[nivel_riesgo]} *{nivel_riesgo}*\n\n"
    )
    msg += ("*Alertas:*\n\n" + "\n\n".join(alertas)) if alertas else "Sin patrones de riesgo detectados."
    logger.info(f"Riesgo [{nombre}]: {nivel_riesgo}")
    return msg

print("Lógica de riesgo lista")

## Informe clínico

In [ ]:
# Genera un informe clínico detallado en formato de texto narrativo
def generar_informe(context):
    datos  = datos_paciente(context)
    nombre = nombre_pac(context)
    rp     = datos['presion']
    rg     = datos['glucosa']
    rm     = datos['medicamentos']
    rs     = datos.get('sintomas', [])
    ra     = datos.get('alertas', [])
    perfil = datos.get('perfil', {})

    if not rp and not rg:
        return None, "Sin mediciones. Registra datos primero."

    inf  = f"INFORME CLÍNICO - {nombre.upper()}\n"
    inf += f"Generado: {ahora()}\n{'═'*36}\n\n"

    # Perfil
    inf += "PERFIL DEL PACIENTE\n"
    inf += f"  Edad: {perfil.get('edad','-')} | Diagnósticos: {perfil.get('diagnosticos','-')}\n"
    inf += f"  Medicación permanente: {perfil.get('medicacion_perm','-')}\n"
    inf += f"  Contacto emergencia: {perfil.get('contacto_emergencia','-')}\n\n"

    # Presión
    if rp:
        vs = [r['sistolica']  for r in rp]
        vd = [r['diastolica'] for r in rp]
        ps, pd = sum(vs)/len(vs), sum(vd)/len(vd)
        hta = [r for r in rp if r['sistolica'] >= UMBRAL_SISTOLICA_MAX or r['diastolica'] >= UMBRAL_DIASTOLICA_MAX]
        inf += f"PRESIÓN ARTERIAL ({len(rp)} mediciones)\n"
        inf += f"Promedio: {ps:.0f}/{pd:.0f} mmHg. "
        inf += ("Promedio elevado - posible hipertensión sostenida. " if ps >= UMBRAL_SISTOLICA_MAX or pd >= UMBRAL_DIASTOLICA_MAX
                else "Promedio dentro del rango aceptable. ")
        inf += f"{len(hta)} episodio(s) hipertensivo(s).\n"
        inf += f"Rango: {min(vs)}-{max(vs)}/{min(vd)}-{max(vd)} mmHg\n"
        for r in rp:
            m = " ⬆HTA" if r['sistolica']>=UMBRAL_SISTOLICA_MAX or r['diastolica']>=UMBRAL_DIASTOLICA_MAX else (
                " ⬇BAJO" if r['sistolica']<UMBRAL_SISTOLICA_MIN or r['diastolica']<UMBRAL_DIASTOLICA_MIN else "")
            inf += f"  [{r['fecha']}] {r['sistolica']}/{r['diastolica']} mmHg{m}\n"
    else:
        inf += "PRESIÓN ARTERIAL: sin registros.\n"
    inf += "\n"

    # Glucosa
    if rg:
        vg   = [r['valor'] for r in rg]
        pg   = sum(vg)/len(vg)
        hiper= [r for r in rg if r['valor'] >= UMBRAL_GLUCOSA_MAX]
        hipo = [r for r in rg if r['valor'] < UMBRAL_GLUCOSA_MIN]
        inf += f"GLUCOSA EN SANGRE ({len(rg)} mediciones)\n"
        inf += f"Promedio: {pg:.0f} mg/dL. "
        if pg >= UMBRAL_GLUCOSA_MAX: inf += "Nivel elevado - revisar con médico. "
        elif pg < UMBRAL_GLUCOSA_MIN: inf += "Nivel bajo - riesgo de hipoglucemia. "
        else: inf += "Nivel dentro del rango normal. "
        if hiper: inf += f"{len(hiper)} episodio(s) hiper. "
        if hipo:  inf += f"{len(hipo)} episodio(s) hipo."
        inf += f"\nRango: {min(vg):.0f}-{max(vg):.0f} mg/dL\n"
        for r in rg:
            m = " ⬆HIPER" if r['valor']>=UMBRAL_GLUCOSA_MAX else (" ⬇HIPO" if r['valor']<UMBRAL_GLUCOSA_MIN else "")
            inf += f"  [{r['fecha']}] {r['valor']:.0f} mg/dL{m}\n"
    else:
        inf += "GLUCOSA: sin registros.\n"
    inf += "\n"

    # Medicamentos
    inf += f"MEDICAMENTOS ({len(rm)} toma(s))\n"
    for r in rm:
        dosis = r.get('dosis','-'); freq = r.get('frecuencia','-')
        inf += f"  [{r['fecha']}] {r['nombre'].capitalize()} {dosis} c/{freq}\n"
    if not rm: inf += "  Sin registros.\n"
    inf += "\n"

    # Síntomas
    inf += f"SÍNTOMAS ({len(rs)} registro(s))\n"
    for s in rs:
        inf += f"  [{s['fecha']}] {s['sintoma']} - {s['intensidad']}\n"
    if not rs: inf += "  Sin síntomas registrados.\n"
    inf += "\n"

    # Alertas
    criticas = [a for a in ra if a['critica']]
    inf += f"ALERTAS ({len(ra)} total, {len(criticas)} crítica(s))\n"
    for a in ra[-10:]:
        tag = " 🚨CRÍTICA" if a['critica'] else ""
        inf += f"  [{a['fecha']}] {a['tipo']}: {a['mensaje']}{tag}\n"
    if not ra: inf += "  Sin alertas.\n"

    inf += f"\n{'─'*36}\n"
    inf += f"Ref: Presión {UMBRAL_SISTOLICA_MIN}-{UMBRAL_SISTOLICA_MAX}/{UMBRAL_DIASTOLICA_MIN}-{UMBRAL_DIASTOLICA_MAX} mmHg | "
    inf += f"Glucosa {UMBRAL_GLUCOSA_MIN}-{UMBRAL_GLUCOSA_MAX} mg/dL (OMS/ADA)\n"

    logger.info(f"Informe generado - {nombre}")
    return inf, None

print("Lógica de informe lista")

## Gráficos

In [ ]:
# Gráficos clínicos de presion arterial y glucosa
def generar_grafico(context):
    datos  = datos_paciente(context)
    nombre = nombre_pac(context)
    rg     = datos.get('glucosa', [])
    rp     = datos.get('presion', [])

    if not rg and not rp:
        return None, "Sin datos para graficar."

    hash_actual = (len(rg), len(rp),
                   (rg[-1]['fecha'] if rg else ''),
                   (rp[-1]['fecha'] if rp else ''))

    cache = datos.get('_grafico_cache', {})
    if cache.get('hash') == hash_actual and cache.get('buf'):
        cache['buf'].seek(0)
        return cache['buf'], None

    n_plots = (1 if rg else 0) + (2 if rp else 0)
    fig, axes = plt.subplots(n_plots, 1, figsize=(11, 5 * n_plots))
    fig.patch.set_facecolor('#F8F9FA')
    if n_plots == 1:
        axes = [axes]
    idx = 0

    # ---------------------- Glucosa ----------------------
    if rg:
        ax   = axes[idx]; idx += 1
        vals = [r['valor'] for r in rg]
        xs   = list(range(len(vals)))
        lsup = max(vals + [UMBRAL_GLUCOSA_MAX + 50])
        ax.axhspan(UMBRAL_GLUCOSA_MAX, lsup, alpha=0.12, color='red',   label=f'Alto (≥{UMBRAL_GLUCOSA_MAX})')
        ax.axhspan(UMBRAL_GLUCOSA_MIN, UMBRAL_GLUCOSA_MAX, alpha=0.08, color='green', label='Normal')
        ax.axhspan(0, UMBRAL_GLUCOSA_MIN, alpha=0.12, color='blue', label=f'Bajo (<{UMBRAL_GLUCOSA_MIN})')
        ax.plot(xs, vals, linewidth=2, marker='o', markersize=7, zorder=5)
        ax.set_xticks(xs)
        ax.set_xticklabels([r['fecha'] for r in rg], rotation=30, ha='right', fontsize=7)
        ax.set_ylabel('mg/dL'); ax.set_ylim(0, lsup)
        ax.set_title('Glucosa', fontsize=12, fontweight='bold')
        ax.legend(fontsize=8, loc='upper left'); ax.grid(True, linestyle=':', alpha=0.5)

    # ---------------------- Presion arterial ----------------------
    if rp:
        fec = [r['fecha']      for r in rp]
        sis = [r['sistolica']  for r in rp]
        dia = [r['diastolica'] for r in rp]
        xs  = list(range(len(sis)))

        ax_s = axes[idx]; idx += 1
        lsup_s = max(sis + [UMBRAL_SISTOLICA_MAX + 20])
        ax_s.axhspan(UMBRAL_SISTOLICA_MAX, lsup_s,              alpha=0.15, color='red',   label=f'Alto (≥{UMBRAL_SISTOLICA_MAX})')
        ax_s.axhspan(UMBRAL_SISTOLICA_MIN, UMBRAL_SISTOLICA_MAX, alpha=0.10, color='green', label='Normal')
        ax_s.axhspan(0, UMBRAL_SISTOLICA_MIN,                    alpha=0.15, color='blue',  label=f'Bajo (<{UMBRAL_SISTOLICA_MIN})')
        ax_s.plot(xs, sis, marker='o', linewidth=2, markersize=7, zorder=5)
        ax_s.set_xticks(xs); ax_s.set_xticklabels(fec, rotation=30, ha='right', fontsize=7)
        ax_s.set_ylabel('mmHg'); ax_s.set_ylim(0, lsup_s)
        ax_s.set_title('Presión sistólica', fontsize=12, fontweight='bold')
        ax_s.legend(fontsize=8, loc='upper left'); ax_s.grid(True, linestyle=':', alpha=0.5)

        ax_d = axes[idx]; idx += 1
        lsup_d = max(dia + [UMBRAL_DIASTOLICA_MAX + 20])
        ax_d.axhspan(UMBRAL_DIASTOLICA_MAX, lsup_d,                alpha=0.15, color='red',   label=f'Alto (≥{UMBRAL_DIASTOLICA_MAX})')
        ax_d.axhspan(UMBRAL_DIASTOLICA_MIN, UMBRAL_DIASTOLICA_MAX, alpha=0.10, color='green', label='Normal')
        ax_d.axhspan(0, UMBRAL_DIASTOLICA_MIN,                     alpha=0.15, color='blue',  label=f'Bajo (<{UMBRAL_DIASTOLICA_MIN})')
        ax_d.plot(xs, dia, linewidth=2, marker='o', markersize=7, zorder=5)
        ax_d.set_xticks(xs); ax_d.set_xticklabels(fec, rotation=30, ha='right', fontsize=7)
        ax_d.set_ylabel('mmHg'); ax_d.set_ylim(0, lsup_d)
        ax_d.set_title('Presión diastólica', fontsize=12, fontweight='bold')
        ax_d.legend(fontsize=8, loc='upper left'); ax_d.grid(True, linestyle=':', alpha=0.5)

    fig.suptitle(f'Paciente: {nombre}',
                 fontsize=11, fontweight='bold', color='#333333', y=0.998)
    plt.tight_layout(pad=3.0)

    buf = BytesIO()
    plt.savefig(buf, format='png', dpi=130, bbox_inches='tight', facecolor='#F8F9FA')
    buf.seek(0)
    plt.close(fig)

    datos['_grafico_cache'] = {'hash': hash_actual, 'buf': buf}
    buf.seek(0)
    return buf, None

print("Lógica de gráfico lista")

# Fase 5 - Motor conversacional

In [ ]:
async def cmd_start(update: Update, context: ContextTypes.DEFAULT_TYPE):
    user_id = update.effective_user.id
    #context.user_data.clear()
    inicializar_estado(context)
    guardar_datos_en_local(context, user_id)

    context.user_data['estado'] = 'menu'
    context.user_data['temp']   = {}

    if hay_paciente_activo(context):
        nombre = context.user_data['paciente_activo']
        await update.message.reply_text(
            f"👋 ¡Hola de nuevo! He recuperado tus datos.\n"
            f"Paciente activo: *{nombre}*",
            parse_mode='Markdown',
            reply_markup=teclado_menu()
        )
    else:
        await update.message.reply_text(
            f"Bienvenido al sistema de monitoreo geriátrico remoto.\n"
            f"No tienes pacientes activos.\n"
            f"Pulsa *👤 Pacientes* para registrar uno",
            parse_mode='Markdown',
            reply_markup=teclado_menu()
        )

In [ ]:
# ============================================================
#  MOTOR CONVERSACIONAL PRINCIPAL
#  Estados:
#   menu | submenu_pacientes | submenu_registrar | submenu_informes
#   esperando_nombre_paciente
#   perfil_edad | perfil_diagnosticos | perfil_medicacion_perm | perfil_contacto
#   esperando_sistolica | esperando_diastolica
#   esperando_glucosa
#   esperando_med_tipo           ← elige dict o libre
#   esperando_med_nombre_libre   ← nombre texto libre
#   esperando_med_dosis          ← dosis texto libre
#   esperando_med_frecuencia     ← frecuencia texto libre
#   esperando_sintoma            ← elige sintoma
#   esperando_sintoma_libre      ← nombre texto libre
#   esperando_intensidad         ← elige intensidad
#   esperando_confirmacion_limpiar
#   reposo | salida
# ============================================================

async def manejar_mensaje(update: Update, context: ContextTypes.DEFAULT_TYPE):
    inicializar_estado(context)
    user_id = update.effective_user.id
    if 'pacientes' not in context.user_data:
        inicializar_estado(context)
        guardar_datos_en_local(context, user_id)
    else:
        inicializar_estado(context)
    texto  = update.message.text.strip()
    estado = context.user_data.get('estado', 'menu')

    # ── Salida ───────────────────────────────────────────────────────────────
    if estado == 'salida':
        context.user_data['estado'] = 'menu'
        await mostrar_menu(update, "👋 ¡Bienvenido de nuevo!", context=context)
        return

    # ── Reposo post-acción ───────────────────────────────────────────────────
    if estado == 'reposo':
        if texto == "📋 Seguir registrando":
            context.user_data['estado'] = 'submenu_registrar'
            await mostrar_submenu_registrar(update, context=context)
        elif texto == "👤 Cambiar paciente":
            context.user_data['estado'] = 'submenu_pacientes'
            await mostrar_submenu_pacientes(update, context)
        elif texto in ("🏠 Volver al menú", "🏠 Abrir menú"):
            context.user_data['estado'] = 'menu'
            await mostrar_menu(update, context=context)
        elif texto == "🚪 Salir por ahora":
            context.user_data['estado'] = 'salida'
            await update.message.reply_text(
                "👋 Listo. Cuando quieras continuar, escribe cualquier mensaje.",
                reply_markup=teclado_despertar()
            )
        else:
            await mostrar_reposo(update)
        return

    # ── Menú principal ───────────────────────────────────────────────────────
    if estado in ('menu', None):
        if texto == "👤 Pacientes":
            context.user_data['estado'] = 'submenu_pacientes'
            await mostrar_submenu_pacientes(update, context)

        elif texto == "📋 Registrar datos":
            if not hay_paciente_activo(context):
                await mostrar_menu(update, "⚠️ Selecciona un paciente primero.", context=context); return
            context.user_data['estado'] = 'submenu_registrar'
            await mostrar_submenu_registrar(update, context=context)

        elif texto == "📊 Informes":
            if not hay_paciente_activo(context):
                await mostrar_menu(update, "⚠️ Selecciona un paciente primero.", context=context); return
            context.user_data['estado'] = 'submenu_informes'
            await mostrar_submenu_informes(update, context=context)

        elif texto == "🗑️ Limpiar registros":
            if not hay_paciente_activo(context):
                await mostrar_menu(update, "⚠️ Selecciona un paciente primero.", context=context); return
            d = datos_paciente(context)
            context.user_data['estado'] = 'esperando_confirmacion_limpiar'
            await update.message.reply_text(
                f"*¿Confirmas borrar los registros de {nombre_pac(context)}?*\n"
                f"{len(d['presion'])} presión | {len(d['glucosa'])} glucosa | "
                f"{len(d['medicamentos'])} medicamentos | {len(d.get('sintomas',[]))} síntomas",
                parse_mode='Markdown',
                reply_markup=ReplyKeyboardMarkup(
                    [["✅ Sí, borrar todo", "❌ No, cancelar"]], resize_keyboard=True, one_time_keyboard=True)
            )

        elif texto == "🚪 Salir por ahora":
            context.user_data['estado'] = 'salida'
            await update.message.reply_text(
                "👋 Hasta pronto. Pulsa el botón cuando quieras continuar.",
                reply_markup=teclado_despertar()
            )
        else:
            await mostrar_menu(update, "No reconocí esa opción. Usa los botones.", context=context)

    # ── Gestión de pacientes ─────────────────────────────────────────────────
    elif estado == 'submenu_pacientes':
        if texto == "➕ Nuevo paciente":
            context.user_data['estado'] = 'esperando_nombre_paciente'
            await update.message.reply_text(
                "¿Cuál es el nombre del paciente?\n_Escribe el nombre completo:_",
                parse_mode='Markdown', reply_markup=ReplyKeyboardRemove()
            )
        elif texto == "⬅️ Volver al menú":
            context.user_data['estado'] = 'menu'
            await mostrar_menu(update, context=context)
        elif texto in context.user_data['pacientes']:
            context.user_data['paciente_activo'] = texto
            context.user_data['estado'] = 'menu'
            d = datos_paciente(context)
            await mostrar_menu(update,
                f"✅ *{texto}* seleccionado - "
                f"{len(d['presion'])}P {len(d['glucosa'])}G {len(d['medicamentos'])}M",
                context=context)
        else:
            await mostrar_submenu_pacientes(update, context, "Opción no reconocida.")

    elif estado == 'esperando_nombre_paciente':
        if len(texto) < 2:
            await update.message.reply_text("Nombre demasiado corto. Escribe el nombre completo:"); return
        creado, nombre_final = crear_paciente(context, texto)
        context.user_data['paciente_activo'] = nombre_final
        context.user_data['estado'] = 'perfil_edad'
        msg = "✅ Paciente creado." if creado else "ℹ️ Paciente ya existe."
        await update.message.reply_text(
            f"{msg}\n\n*Completemos el perfil de {nombre_final}*\n\n"
            "¿Cuántos años tiene el paciente?",
            parse_mode='Markdown', reply_markup=ReplyKeyboardRemove()
        )

    # ── Perfil: campos secuenciales ──────────────────────────────────────────
    elif estado == 'perfil_edad':
        try:
            edad = int(texto)
            if not (0 < edad < 130): raise ValueError
            datos_paciente(context)['perfil']['edad'] = str(edad)
            context.user_data['estado'] = 'perfil_diagnosticos'
            await update.message.reply_text(
                "¿Cuáles son sus diagnósticos crónicos?\n_ej: HTA, Diabetes tipo 2_",
                parse_mode='Markdown')
        except ValueError:
            await update.message.reply_text("Ingresa una edad válida (número entre 1 y 129):")

    elif estado == 'perfil_diagnosticos':
        datos_paciente(context)['perfil']['diagnosticos'] = texto
        context.user_data['estado'] = 'perfil_medicacion_perm'
        await update.message.reply_text(
            "¿Cuál es su medicación permanente?\n_ej: Metformina 500mg, Losartan 50mg_",
            parse_mode='Markdown')

    elif estado == 'perfil_medicacion_perm':
        datos_paciente(context)['perfil']['medicacion_perm'] = texto
        context.user_data['estado'] = 'perfil_contacto'
        await update.message.reply_text(
            "¿Nombre y teléfono del contacto de emergencia?\n_ej: María López - 987654321_",
            parse_mode='Markdown')

    elif estado == 'perfil_contacto':
        # Valida que el texto contenga exactamente 9 dígitos consecutivos (teléfono)
        telefono_match = _re.search(r'\b(\d{9})\b', texto)
        if not telefono_match:
            await update.message.reply_text(
                "El teléfono debe tener exactamente 9 dígitos.\n"
                "Formato: Eduardo - 987654321\n"
                "_Vuelve a escribir el contacto de emergencia:_",
                parse_mode='Markdown'
            )
            return   # no avanza de estado
        datos_paciente(context)['perfil']['contacto_emergencia'] = texto
        context.user_data['estado'] = 'menu'
        user_id = update.effective_user.id
        guardar_datos_en_drive(context, user_id)
        await mostrar_menu(update,
            f"✅ Perfil de *{nombre_pac(context)}* completado.",
            context=context)

    # ── Registrar datos ──────────────────────────────────────────────────────
    elif estado == 'submenu_registrar':
        if texto == "🩺 Presión arterial":
            context.user_data['estado'] = 'esperando_sistolica'
            await update.message.reply_text(
                f"*Presión arterial - {nombre_pac(context)}*\n"
                "━━━━━━━━━━━━━━━━━━━\n"
                "Paso 1/2 - ¿Presión *sistólica*? (número, ej: 120)",
                parse_mode='Markdown', reply_markup=ReplyKeyboardRemove())

        elif texto == "🩸 Glucosa":
            context.user_data['estado'] = 'esperando_glucosa'
            await update.message.reply_text(
                f"*Glucosa - {nombre_pac(context)}*\n"
                "¿Nivel de glucosa en sangre (mg/dL)? (número, ej: 145)",
                parse_mode='Markdown', reply_markup=ReplyKeyboardRemove())

        elif texto == "💊 Medicamento":
            context.user_data['estado'] = 'esperando_med_tipo'
            await update.message.reply_text(
                f"*Medicamento - {nombre_pac(context)}*\n"
                "Selecciona o elige «Otro»:",
                parse_mode='Markdown', reply_markup=teclado_medicamentos_dict())

        elif texto == "🤒 Síntomas":
            context.user_data['estado'] = 'esperando_sintoma'
            await update.message.reply_text(
                f"*Síntomas - {nombre_pac(context)}*\n"
                "¿Qué síntoma presenta el paciente?",
                parse_mode='Markdown', reply_markup=teclado_sintomas())

        elif texto == "⬅️ Volver al menú":
            context.user_data['estado'] = 'menu'
            await mostrar_menu(update, context=context)
        else:
            await mostrar_submenu_registrar(update, "Opción no reconocida.", context=context)

    # ── Presión sistólica ────────────────────────────────────────────────────
    elif estado == 'esperando_sistolica':
        try:
            s = int(texto)
            if not (40 <= s <= 300): raise ValueError("rango")
            context.user_data['temp']['sist'] = s
            context.user_data['estado'] = 'esperando_diastolica'
            await update.message.reply_text(
                f"Sistólica: *{s} mmHg* ✓\nPaso 2/2 - ¿Presión *diastólica*? (ej: 80)",
                parse_mode='Markdown')
        except ValueError as e:
            msg = "⚠️ Rango inválido (40–300 mmHg)." if "rango" in str(e) else "Ingresa solo un número entero."
            await update.message.reply_text(msg)

    # ── Presión diastólica ────────────────────────────────────────────────────
    elif estado == 'esperando_diastolica':
        try:
            d = int(texto)
            if not (30 <= d <= 160): raise ValueError("rango")
            s = context.user_data['temp'].get('sist')
            if s is None:
                context.user_data['estado'] = 'esperando_sistolica'
                await update.message.reply_text("Error de flujo. Reingresa la sistólica:")
                return
            user_id = update.effective_user.id
            await update.message.reply_text("Registrando…")
            conf = guardar_presion(context, s, d, user_id)
            context.user_data['temp'].pop('sist', None)
            await update.message.reply_text(conf, parse_mode='Markdown')
            await mostrar_reposo(update, "✅ Guardado.")
            context.user_data['estado'] = 'reposo'
        except ValueError as e:
            msg = "Rango inválido (30–160 mmHg)." if "rango" in str(e) else "Ingresa un número entero."
            await update.message.reply_text(msg)

    # ── Glucosa ──────────────────────────────────────────────────────────────
    elif estado == 'esperando_glucosa':
        try:
            v = float(texto.replace(',', '.'))
            if not (20 <= v <= 600):
                raise ValueError("rango")

            user_id = update.effective_user.id
            await update.message.reply_text("Registrando…")
            conf = guardar_glucosa(context, v, user_id)

            await update.message.reply_text(conf, parse_mode='Markdown')
            await mostrar_reposo(update, "✅ Guardado.")
            context.user_data['estado'] = 'reposo'

        except ValueError as e:
            msg = "⚠️ Rango inválido (20–600 mg/dL)." if "rango" in str(e) else "Ingresa solo un número."
            await update.message.reply_text(msg)

    # ── Medicamento: elección tipo ───────────────────────────────────────────
    elif estado == 'esperando_med_tipo':
        if texto == "⬅️ Volver al menú":
            context.user_data['estado'] = 'submenu_registrar'
            await mostrar_submenu_registrar(update, context=context); return

        if texto == "➕ Otro medicamento":
            context.user_data['estado'] = 'esperando_med_nombre_libre'
            await update.message.reply_text(
                "¿Cuál es el nombre del medicamento?"
                "Ejemplo: 1 Losartan o 2 Metformina",
                reply_markup=ReplyKeyboardRemove()); return

        nombre_c = texto.strip().lower()
        if nombre_c in MEDICAMENTOS_DICT:
            try:
                user_id = update.effective_user.id
                total, fecha_registro = guardar_medicamento_dict(context, nombre_c, user_id)
                await update.message.reply_text(
                    f"✅ *{nombre_c.capitalize()}* confirmado [{fecha_registro}]",
                    parse_mode='Markdown')
                await update.message.reply_text(
                    "¿Registrar otro medicamento o volver?",
                    reply_markup=teclado_confirmar_med())
                context.user_data['estado'] = 'esperando_otro_med'
            except Exception as ex:
                logger.error(ex)
                await update.message.reply_text("Error al guardar. Intenta de nuevo.",
                    reply_markup=teclado_medicamentos_dict())
        else:
            await update.message.reply_text(
                "Selecciona un medicamento de la lista o elige «➕ Otro».",
                reply_markup=teclado_medicamentos_dict())

    # ── Medicamento libre: nombre ────────────────────────────────────────────
    elif estado == 'esperando_med_nombre_libre':
        # Formato obligatorio: número seguido de texto alfabético
        # Válido: "1 Losartan", "2 Metformina"   Inválido: "Losartan", "@Insulina"
        if not _re.match(r'^\d+\s+[A-Za-záéíóúÁÉÍÓÚñÑ]', texto):
            await update.message.reply_text(
                "Formato inválido.\n"
                "Escribe un número seguido del nombre.\n"
                "Ejemplo: 1 Losartan o 2 Metformina",
                parse_mode='Markdown'
            )
            return   # no avanza de estado
        context.user_data['temp']['med_nombre'] = texto
        context.user_data['estado'] = 'esperando_med_dosis'
        await update.message.reply_text(
            f"Medicamento: *{texto}*\n¿Cuál es la dosis? (ej: 500 mg, 10 UI)",
            parse_mode='Markdown'
        )

    elif estado == 'esperando_med_dosis':
        if len(texto) < 1:
            await update.message.reply_text("Escribe la dosis (ej: 500mg):"); return
        context.user_data['temp']['med_dosis'] = texto
        context.user_data['estado'] = 'esperando_med_frecuencia'
        await update.message.reply_text(
            f"Dosis: *{texto}*\n¿Con qué frecuencia? (ej: cada 8 horas, 1 vez al día)",
            parse_mode='Markdown')

    elif estado == 'esperando_med_frecuencia':
        if len(texto) < 1:
            await update.message.reply_text("Escribe la frecuencia (ej: cada 12 horas):"); return
        nombre  = context.user_data['temp'].get('med_nombre', '?')
        dosis   = context.user_data['temp'].get('med_dosis', '?')
        freq    = texto
        user_id = update.effective_user.id
        total, fecha = guardar_medicamento_libre(context, nombre, dosis, freq, user_id)
        context.user_data['temp'].pop('med_nombre', None)
        context.user_data['temp'].pop('med_dosis',  None)
        await update.message.reply_text(
            f"✅ *{nombre.capitalize()}* {dosis} c/{freq} - [{fecha}] toma #{total}",
            parse_mode='Markdown')
        await update.message.reply_text(
            "¿Registrar otro medicamento o volver?",
            reply_markup=teclado_confirmar_med())
        context.user_data['estado'] = 'esperando_otro_med'

    # ── Confirmar si registra otro med ──────────────────────────────────────
    elif estado == 'esperando_otro_med':
        if texto == "✅ Registrar otro":
            context.user_data['estado'] = 'esperando_med_tipo'
            await update.message.reply_text(
                "Selecciona el siguiente medicamento:",
                reply_markup=teclado_medicamentos_dict())
        else:
            await mostrar_reposo(update, "✅ Medicamentos guardados.")
            context.user_data['estado'] = 'reposo'

    # ── Síntomas ─────────────────────────────────────────────────────────────
    elif estado == 'esperando_sintoma':
        if texto == "⬅️ Volver al menú":
            context.user_data['estado'] = 'submenu_registrar'
            await mostrar_submenu_registrar(update, context=context); return
        if texto == "➕ Otro síntoma":
            context.user_data['estado'] = 'esperando_sintoma_libre'
            await update.message.reply_text(
                "¿Cuál es el síntoma?", reply_markup=ReplyKeyboardRemove()); return
        if texto in SINTOMAS_COMUNES:
            context.user_data['temp']['sintoma'] = texto
            context.user_data['estado'] = 'esperando_intensidad'
            await update.message.reply_text(
                f"Síntoma: *{texto}*\n¿Cuál es la intensidad?",
                parse_mode='Markdown', reply_markup=teclado_intensidad())
        else:
            await update.message.reply_text(
                "Selecciona un síntoma de la lista o «➕ Otro».",
                reply_markup=teclado_sintomas())

    elif estado == 'esperando_sintoma_libre':
        if len(texto) < 2:
            await update.message.reply_text("Describe el síntoma con más detalle:"); return
        context.user_data['temp']['sintoma'] = texto
        context.user_data['estado'] = 'esperando_intensidad'
        await update.message.reply_text(
            f"Síntoma: *{texto}*\n¿Cuál es la intensidad?",
            parse_mode='Markdown', reply_markup=teclado_intensidad())

    elif estado == 'esperando_intensidad':
        if texto == "⬅️ Cancelar":
            context.user_data['estado'] = 'submenu_registrar'
            await mostrar_submenu_registrar(update, context=context); return
        if texto not in ("🟡 Leve", "🟠 Moderado", "🔴 Severo"):
            await update.message.reply_text(
                "Selecciona la intensidad usando los botones.",
                reply_markup=teclado_intensidad()); return
        sintoma = context.user_data['temp'].get('sintoma', '?')
        user_id = update.effective_user.id
        fecha = guardar_sintoma(context, sintoma, texto, user_id)
        context.user_data['temp'].pop('sintoma', None)
        alerta_extra = ""
        if texto == "🔴 Severo":
            alerta_extra = "\n🚨 *Síntoma severo registrado como alerta crítica.*"
        await update.message.reply_text(
            f"✅ *{sintoma}* ({texto}) registrado [{fecha}]{alerta_extra}",
            parse_mode='Markdown')
        await mostrar_reposo(update, "")
        context.user_data['estado'] = 'reposo'

    # ── Informes ─────────────────────────────────────────────────────────────
    elif estado == 'submenu_informes':
        if texto == "⚠️ Evaluación de riesgo":
            await update.message.reply_text(evaluar_riesgo(context), parse_mode='Markdown')
            # Permanece en submenu_informes - no forzar reposo
            await mostrar_submenu_informes(update, context=context)

        elif texto == "📄 Generar informe completo":
            # Respuesta inmediata al usuario (corrección 4)
            await update.message.reply_text("Procesando informe…", reply_markup=ReplyKeyboardRemove())
            inf, err = generar_informe(context)
            if err:
                await update.message.reply_text(err)
                await mostrar_submenu_informes(update, context=context)
                return
            await update.message.reply_text(f"```\n{inf}\n```", parse_mode='Markdown')
            buf, err_g = generar_grafico(context)
            if err_g:
                await update.message.reply_text(
                    f"_(Gráfico no disponible: {err_g})_", parse_mode='Markdown'
                )
            else:
                await update.message.reply_photo(
                    photo=InputFile(buf, filename='grafico_clinico.png')
                )
            # Permanece en submenu_informes - el usuario decide cuándo salir
            await mostrar_submenu_informes(update, context=context)

        elif texto == "👤 Ver perfil del paciente":
            await update.message.reply_text(texto_perfil(context), parse_mode='Markdown')
            await mostrar_submenu_informes(update, context=context)

        elif texto == "⬅️ Volver al menú":
            context.user_data['estado'] = 'menu'
            await mostrar_menu(update, context=context)

        else:
            # Cualquier otro mensaje: permanecer en informes, no romper flujo
            await mostrar_submenu_informes(update,
                "Usa los botones para consultar.", context=context)

    # ── Limpiar registros ─────────────────────────────────────────────────────
    elif estado == 'esperando_confirmacion_limpiar':
        tl = texto.lower()
        if "sí" in tl or "si" in tl or "borrar" in tl:
            d = datos_paciente(context)
            np_, ng, nm, ns = len(d['presion']), len(d['glucosa']), len(d['medicamentos']), len(d.get('sintomas',[]))
            d['presion']=[]; d['glucosa']=[]; d['medicamentos']=[]; d['sintomas']=[]; d['alertas']=[]
            context.user_data['estado'] = 'menu'
            logger.info(f"Registros de {nombre_pac(context)} limpiados.")
            guardar_datos_en_drive(context, user_id)
            await update.message.reply_text(
                f"*Registros eliminados - {nombre_pac(context)}*\n"
                f"{np_}P {ng}G {nm}M {ns}S borrados.",
                parse_mode='Markdown')
        else:
            context.user_data['estado'] = 'menu'
            await update.message.reply_text("Cancelado. Los registros no fueron borrados.")
        await mostrar_menu(update, context=context)

    # ── Fallback ─────────────────────────────────────────────────────────────
    else:
        context.user_data['estado'] = 'menu'
        await mostrar_menu(update, "Error en el flujo. Volviendo al menú.", context=context)

print("Motor conversacional listo")

# Fase 6 - Despliegue

In [ ]:
def Iniciar():
  try:
    # Guarda todo context.user_data en un archivo .pkl en disco
    persistence = PicklePersistence(filepath="datos_pacientes.pkl")
    app = (
        ApplicationBuilder()
        .token(TOKEN)
        .persistence(persistence)
        .build()
    )
    app.add_handler(CommandHandler("start", cmd_start))
    app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, manejar_mensaje))

    print("✅ Bot en marcha...✅")
    app.run_polling(stop_signals=None)
  except KeyboardInterrupt:
        pass # Aquí manejamos el cierre silencioso
  except Exception as e:
      print("❌ Bot apagado...❌")

In [ ]:
Iniciar()